# ДЗ1


Задание состоит из четырех частей.

| Часть | Тема |
|---|---|
| A | Bag of Little Bootstraps: от чего зависит валидность выводов | 
| B | Теорема Фриша–Во–Ловелла и её следствия для CUPED и стратификации | 
| C | Медиана: бутстрэп без ресэмплинга (Spotify, Evan Miller, Price–Bonett) | 
| D | Выбор "случайных" групп | 

**Что сдавать.** Этот же ноутбук с заполненными решениями. Ответ вида «получилось вот такое число» принимать не хочется, хотелось бы объяснение, почему получилось именно так.

**Правила.** Пишите (редактируйте) нормальный код, который можно читать без таблетки нурофена.

---
# Часть A. Bag of Little Bootstraps 

## Постановка

Обычный бутстрэп на выборке размера $n$ требует построить $B$ псевдовыборок, каждая размера $n$. Стоимость $O(Bn)$, и при $n \sim 10^8$ это уже неподъёмно. Каждая псевдовыборка содержит в среднем $0{,}632n$ различных точек, то есть почти столько же данных, сколько исходная выборка. 

**Bag of Little Bootstraps** (Kleiner, Talwalkar, Sarkar, Jordan, 2014) устроен так:

1. выбрать размер подвыборки $b = n^{\gamma}$, где $\gamma \in [0{,}5;\ 1)$;
2. взять $s$ подвыборок размера $b$ без возвращения;
3. внутри каждой подвыборки $j$ сделать $r$ перевзвешиваний: веса $(n_1, \dots, n_b) \sim \mathrm{Multinomial}\bigl(n,\ (1/b, \dots, 1/b)\bigr)$, посчитать взвешенную статистику $\hat\theta^{*}_{j,k}$;
4. по $r$ репликациям внутри подвыборки посчитать нужную характеристику качества $\xi_j$ (ширину доверительного интервала, стандартную ошибку, квантили);
5. усреднить $\xi_j$ по $s$ подвыборкам.

На шаге 3 сумма весов равна $n$, а не $b$. Мы физически держим в памяти только $b$ точек, но статистика считается так, как если бы выборка была размера $n$. Поэтому разброс бутстрэп-распределения внутри подвыборки имеет правильный масштаб $\sigma/\sqrt{n}$, а не $\sigma/\sqrt{b}$.

Стоимость одной подвыборки $O(rb)$ вместо $O(Bn)$, и подвыборки обрабатываются независимо, то есть задача параллелизуется.

**Что предстоит выяснить.** Алгоритм имеет три параметра: $\gamma$, $s$, $r$. Вопрос задания: при каких их значениях выводы BLB совпадают с выводами обычного бутстрэпа, и от чего эта граница зависит.

**Что делать-то**:


## Пункт 1

Напишите BLB-функцию. Она должна возвращать сразу четыре величины:

* `lo_avg`, `hi_avg` — усреднённые по подвыборкам границы интервала
* `se` — усреднённая по подвыборкам стандартная ошибка бутстрэп-распределения;
* `b` — фактический размер подвыборки.

Очевидно, та же функция может дать вам и обычный бутстреп :)

## Пункт 2 

Возьмите сетку $\gamma$, фиксируя $s$ и $r$. 
Столбцы: $\gamma$, $b$, ширина интервала BLB, относительная ошибка ширины относительно обычного бутстрэпа, время счёта.

Относительная ошибка: $\bigl(w_{\mathrm{BLB}} - w_{\mathrm{boot}}\bigr) / w_{\mathrm{boot}}$.

## Пункт 3

Попробуйте теперь посмотреть, как влияют на результат $s$ и $r$ (таким же образом - взять и выбрать сетку параметров, пройтись по ней и посмотреть, что получится)


## Финал

По результатам вашего собственного моделирования скажите что-то про "оптимальные" параметры для BLB и "почему так".

## P.S.

В качестве тест-кейса не забыть попробовать какое-то распределение "с тяжелым хвостом" типа логнормального.

---
# Часть B. Теорема Фриша–Во–Ловелла (25 баллов)

## Постановка

Пусть оценивается регрессия
$$y = X_1 \beta_1 + X_2 \beta_2 + \varepsilon.$$

Обозначим за $M_2 = I - X_2 (X_2^{\top} X_2)^{-1} X_2^{\top}$ оператор взятия остатков от проекции на $X_2$. Теорема Фриша–Во–Ловелла утверждает, что $\hat\beta_1$ из полной регрессии в точности совпадает с оценкой из короткой регрессии остатков на остатки:
$$\hat\beta_1 = \bigl((M_2 X_1)^{\top} M_2 X_1\bigr)^{-1} (M_2 X_1)^{\top} M_2 y.$$

Содержательно: «очистить $y$ и $X_1$ от влияния $X_2$, а потом связать очищенные величины» и «оценить всё сразу» — это одно и то же. Теорема доказана Фришем и Во в 1933 году, за работы этого цикла Фриш получил первую Нобелевскую премию по экономике.

Из неё следует довольно многое: и CUPED, и стратификация — это частные случаи регрессии с дополнительными регрессорами.

## Пункт 1.

Численно проверьте теорему, используя OLS.

In [ ]:
# КОД

## Пункт 2

Проверьте , что CUPED эквивалентен регрессии с одной доп.переменной

In [8]:
# КОД

---
# Часть C. Медиана: бутстрэп без ресэмплинга

## Зачем

Обычный процентильный бутстрэп медианы требует $B$ раз отсортировать (или хотя бы частично упорядочить) выборку размера $n$. При $n \sim 10^8$ и $B = 10^3$ это недоступно.

Но у медианы есть особенность: бутстрэп-медиана всегда равна какой-то **порядковой статистике исходной выборки**. Значит, бутстрэп-распределение медианы полностью описывается распределением **индекса** $i$, при котором $x_{(i)}$ оказывается медианой псевдовыборки. Достаточно один раз отсортировать выборку и знать это распределение индекса.

Разберём три способа его получить.

**1. Spotify (Schultzberg, Ankargren, 2022).** Эмпирически обнаружено и предложено как гипотеза: индекс бутстрэп-квантиля $q$ приближённо распределён как $\mathrm{Bin}(n+1,\ q)$. Отсюда интервал: взять квантили биномиального распределения и вернуть соответствующие порядковые статистики.

**2. Evan Miller (2022).** Для медианы и пуассоновского бутстрэпа выведено **точное** распределение индекса. Пусть $z = 2\sqrt{(i-1)(n-i)}$ и
$$I_{\nu} = I_{\nu}(z), \qquad I^{*}_{\nu} = \left[\left(\frac{i-1}{n-i}\right)^{\nu/2} + \left(\frac{n-i}{i-1}\right)^{\nu/2}\right] I_{\nu},$$
где $I_{\nu}$ — модифицированная функция Бесселя первого рода. Тогда
$$\Pr\bigl(x_{(i)} = \hat\tau_{1/2}\bigr) = e^{-n}\left[c_0 I_0 + \sum_{k \ge 1} c_k I^{*}_k\right], \qquad c_0 = e - 1, \quad c_k = \frac{1}{2\,k!} + e - \sum_{s=0}^{k}\frac{1}{s!}.$$

**3. Price–Bonett (2001, 2002).** Подход не бутстрэповский, а аналитический. Берётся точный непараметрический интервал для медианы по порядковым статистикам, и его ширина пересчитывается в стандартную ошибку:
$$\widehat{\mathrm{se}}(\hat\eta) = \frac{x_{(n-c+1)} - x_{(c)}}{2 z_{p}}, \qquad c = \left[\frac{n+1}{2} - \sqrt{n}\right], \quad p = 2\Pr\bigl(\mathrm{Bin}(n, 1/2) \le c - 1\bigr), \quad z_p = \Phi^{-1}(1 - p/2).$$
Дальше интервал строится обычным образом, а для разности медиан складываются дисперсии.

Источники: [arXiv:2202.10992](https://arxiv.org/abs/2202.10992), [evanmiller.org](https://www.evanmiller.org/bootstrapping-sample-medians.html), [Price & Bonett (2002)](https://www.tandfonline.com/doi/abs/10.1080/00949650212140).

## Пункты 1,2,3

Написать реализацию построения доверительного интервала каждым из методов :) Если ваш LL

# Часть D. Выбор случайных групп

Вам дан датасет из 15000 магазинов. Задача - многократно случайным образом выбрать из этих 15000 две группы по 1500 и проверить, насколько они "статистически эквивалентны". Вы будете брать `hash(store_id+salt)` и перебирать разный `salt`, чтоб получать разные выборки. Уверен, между группами часто будет получаться довольно большая разница. 

Тут надо будет ответить на вопрос "а насколько вменяемым" получается такой способ разбиения.

In [4]:
import pandas as pd
link = 'https://github.com/pileyan/statistics_dda/raw/refs/heads/master/hw1/data.xlsx'

store_data = pd.read_excel(link)

In [7]:
store_data.store_id.nunique()

15000

# Комментарий к выполнению заданий

1) Огромная просьба, напишите ответы по-человечески. Ну и если когда будете код генерировать бездушной машиной, следите за тем, чтобы код получался +- эффективным (это напрямую проверяться не будет, но это хороший тон). 

2) Текстовую часть ответа нужно будет причесать в нормальном виде (например, как если бы у вас это попросили сделать на работе). От слишком сурового нейрослопа вида "да, вы правы, у меня нет оправданий на этот счет. Попробуем разобраться еще раз" меня может стошнить. 
